### Classical ML

Non-tree supervised methods. Tree-based methods (bagging, boosting) live in their own notebooks (`bagging.ipynb`, `boosting.ipynb`), `decision-tree.ipynb` is the standalone foundational base learner. Covers Logistic Regression, Linear Regression, SVM, KNN, Naive Bayes.

## Logistic Regression

## Logistic Regression, From Scratch

See also: [05-xgboost-from-scratch.ipynb](xgboost-from-scratch.ipynb) (same p-y gradient extended to boosted trees).

Plain terms: start with a random guess for the weights. For every point, check how wrong that guess is, then nudge the weights a small step in the direction that makes the guess less wrong. Repeat until the wrongness stops shrinking. Sigmoid turns the raw guess into a probability; gradient descent is just this repeated small-correction process.

1. Probability, odds, log-odds: $p=\text{successes}/\text{total}\in[0,1]$. Odds $=p/(1-p)\in[0,\infty)$. Log-odds $=\ln(\text{odds})\in(-\infty,\infty)$.

| state | p | odds | log-odds |
|---|---|---|---|
| impossible | 0.0 | 0 | $-\infty$ |
| unlikely | 0.2 | 0.25 | -1.386 |
| coin toss | 0.5 | 1 | 0.0 |
| likely | 0.8 | 4.0 | +1.386 |
| certain | 1.0 | $\infty$ | $+\infty$ |

2. Why log-odds, not $p$, as the linear target: $z=w^Tx+b$ is unconstrained in $(-\infty,\infty)$; setting $p=z$ directly breaks the $[0,1]$ bound. Setting $\ln(p/(1-p))=z$ keeps domains matched. $w_j$ = change in log-odds per unit $x_j$; $e^{w_j}$ = odds ratio.
3. Sigmoid, derived from log-odds: $p/(1-p)=e^z \Rightarrow p=e^z(1-p) \Rightarrow p+pe^z=e^z \Rightarrow p=\frac{e^z}{1+e^z}=\frac{1}{1+e^{-z}}$. Squashes any real $z$ into $(0,1)$: at $z=0$, $p=0.5$; as $z\to\pm\infty$, $p\to1$ or $0$. Symmetry $\sigma(-z)=1-\sigma(z)$; derivative $\sigma'(z)=\sigma(z)(1-\sigma(z))=p(1-p)$.
4. MLE, single point: $P(y_i|x_i)=p_i^{y_i}(1-p_i)^{1-y_i}$, collapses to $p_i$ when $y_i=1$, $(1-p_i)$ when $y_i=0$. Joint likelihood (i.i.d.): $L(w)=\prod p_i^{y_i}(1-p_i)^{1-y_i}$. Probability fixes $w$, varies $y$ ("how likely are these outcomes"); likelihood fixes $(x,y)$, varies $w$ ("how plausible are these weights").
5. Log-loss, from MLE: log turns the product into a sum, avoids underflow: $\ell(w)=\sum[y_i\ln p_i+(1-y_i)\ln(1-p_i)]$. Gradient descent minimizes, MLE maximizes, negate and average: $L(w)=-\frac{1}{N}\sum[y_i\ln p_i+(1-y_i)\ln(1-p_i)]$.
6. Gradient, chain rule: $\frac{\partial L_i}{\partial w_j}=\frac{\partial L_i}{\partial p_i}\cdot\frac{\partial p_i}{\partial z_i}\cdot\frac{\partial z_i}{\partial w_j}=\left[\frac{p_i-y_i}{p_i(1-p_i)}\right]\cdot[p_i(1-p_i)]\cdot x_{ij}=(p_i-y_i)x_{ij}$. The $p(1-p)$ terms cancel.
7. Batch gradient: $\nabla L=\frac{1}{N}X^T(p-y)$. Update: $w\leftarrow w-\eta\nabla L$. $p>y$ (overestimating) $\Rightarrow$ gradient positive $\Rightarrow$ $w$ decreases; $p<y$ (underestimating) $\Rightarrow$ gradient negative $\Rightarrow$ $w$ increases.


#### Concept note: multinomial logistic regression

K classes → one weight vector per class (unlike binary logreg's single vector,
since p(other classes) isn't just 1 - p(one class) anymore).

- Score:   z = W·x            (one raw score per class)
- Predict: p = softmax(z) = exp(z_k) / Σ exp(z_j)   → probabilities summing to 1
- Loss:    cross-entropy = -Σ y_k·log(p_k)          (y = one-hot true label)
- Gradient (closed form for softmax + cross-entropy): dL/dz = p - y
- Update:  W = W - lr · (p - y)ᵀ·x, repeated up to max_iter times, or until it converges

Shapes here: X (1400, 2000) · Wᵀ (2000, 10) → Z (1400, 10) → softmax row-wise → P (1400, 10)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

n = 200
X0 = rng.normal(loc=[-2, -2], scale=1.0, size=(n, 2))
X1 = rng.normal(loc=[2, 2], scale=1.0, size=(n, 2))
X = np.vstack([X0, X1])
y = np.hstack([np.zeros(n), np.ones(n)])

plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", alpha=0.6)
plt.title("toy data")
plt.show()

# squashes any real z into (0,1); derivative = p(1-p) -> steep at 0.5, flat at extremes
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# negative log-likelihood of a Bernoulli label (from MLE) -- punishes confident-and-wrong hard
def log_loss(y, p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)  # guards log() against log(0)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))


In [ ]:
def train_logistic_regression(X, y, lr=0.1, n_iters=1000):
    n_samples, n_features = X.shape
    w = np.zeros(n_features)   # start with no opinion, all weights zero
    b = 0.0
    losses = []

    for i in range(n_iters):
        z = X @ w + b               # the linear part: w·x + b, for every row at once
        p = sigmoid(z)               # squash into probabilities

        loss = log_loss(y, p)
        losses.append(loss)

        # (p-y)*x is the derived per-point gradient; dividing by n_samples averages
        # instead of sums, so the update size doesn't depend on how many rows we have
        dw = (X.T @ (p - y)) / n_samples
        db = np.mean(p - y)          # same gradient, but for the bias (x is implicitly 1 here)

        w -= lr * dw                 # step opposite the gradient, that's what makes loss go down
        b -= lr * db

    return w, b, losses

w, b, losses = train_logistic_regression(X, y, lr=0.1, n_iters=1000)

plt.plot(losses)
plt.xlabel("iteration")
plt.ylabel("loss")
plt.title("training loss over time")
plt.show()

print("learned w:", w)
print("learned b:", b)


In [ ]:
from sklearn.linear_model import LogisticRegression

sklearn_model = LogisticRegression()
sklearn_model.fit(X, y)

print("sklearn w:", sklearn_model.coef_[0])
print("sklearn b:", sklearn_model.intercept_[0])

# ours:    w=[1.936, 1.977], b=0.323
# sklearn: w=[1.884, 1.887], b=0.489
# same sign, same rough ratio between the two features -> confirms our from-scratch
# gradient derivation is correct. Not identical because sklearn adds L2 regularization
# by default and uses a different optimizer, not plain gradient descent.


1. Vanishing gradient with MSE
With MSE, loss = (1/n) Σ(p - y)². Its gradient w.r.t. weights includes a sigmoid'(z) term (chain rule through the sigmoid):

dL/dw = (1/n) Σ 2(p-y) * sigmoid'(z) * x

sigmoid'(z) = p(1-p), which is near 0 when p is near 0 or 1 (saturated regions). So when the model is confidently wrong (predicts p=0.99 but true y=0), the gradient is tiny — the model barely updates, exactly when it needs the biggest correction. Learning stalls.

With BCE, the sigmoid'(z) term cancels out algebraically in the gradient derivation, leaving simply (p - y). Gradient magnitude scales directly with the error — confidently wrong predictions produce large gradients, so the model corrects fast. This is the core practical reason.

2. Convexity from Maximum Likelihood
Logistic regression models y as Bernoulli(p). The likelihood of the data is Π p^y (1-p)^(1-y). Taking negative log-likelihood gives exactly the BCE loss. This connection matters because:

BCE (as a function of w) is convex — one global minimum, gradient descent converges reliably.
MSE composed with sigmoid is non-convex — multiple local minima/saddle points, harder optimization landscape.

3. Probabilistic interpretation
BCE directly penalizes based on how "surprised" the model is (information-theoretic: cross-entropy measures divergence between true label distribution and predicted distribution). MSE treats the output as a continuous regression target, which doesn't match the semantics of a probability.

Multiclass extension: softmax + categorical cross-entropy — same vanishing gradient logic applies.

L2 regularization: add λ||w||² to loss — how does the gradient update change? (grad_w += (λ/n) * w)

## Likely Questions

1. Why sigmoid, not some other squashing function? It's the inverse of the logit, the natural link between "linear in log-odds" and "output as a probability." Pairs with log-loss for a clean single-term gradient, $(p-y)x$; a different squashing function wouldn't cancel that cleanly.
2. Why log-loss instead of MSE? Falls directly out of MLE for a Bernoulli label, not chosen by convention. MSE with a sigmoid output is non-convex (gradient descent can get stuck); log-loss with sigmoid is convex, one global minimum guaranteed.
3. How do you interpret a coefficient? Change in log-odds of the positive class per unit increase in that feature, holding others fixed. $e^w$ gives an odds ratio, e.g. $e^w=1.5$ means a 50% increase in the odds of fraud per unit.
4. Why can't logistic regression capture "amount high AND category risky" the way a rule can? $z$ is a plain sum of independently-weighted features, no term for two features acting together unless engineered explicitly. A hand-written AND rule beat logistic regression at equal recall on the fraud project.
5. Why does logistic regression need feature scaling but trees don't? Coefficients are tied to feature magnitude, an unscaled large-range feature dominates the loss from scale alone. Trees split on "is this value above X," scale-invariant.
6. How do you handle severe class imbalance? Weight each sample's loss inversely to its class frequency (`class_weight="balanced"`), or resample. Don't trust a default 0.5 threshold, pick the operating threshold from the actual cost of each error type.
7. Is logistic regression linear? Yes where it matters, the decision boundary ($p=0.5$) is a linear hyperplane, linear in log-odds. Sigmoid itself is nonlinear, but that's just the link function.
8. L1 vs L2 regularization? Both shrink coefficients toward zero. L1 (Lasso) can push some exactly to zero, automatic feature selection. L2 (Ridge, sklearn default) shrinks smoothly, rarely to exactly zero.


#### Implementation appendix: numerically stable softmax

In [ ]:
import numpy as np

def softmax(z: np.ndarray) -> np.ndarray:
    """
    Computes a numerically stable softmax function.
    Args:
        z (np.ndarray): A 1D or 2D array of raw scores (logits).
    Returns:
        np.ndarray: The probability distribution.
    """
    # Ensure z is at least 2D for consistent max operation
    if z.ndim == 1:
        z = z.reshape(1, -1)

    # The stability trick: subtract the max for each sample
    max_z = np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(z - max_z)

    # Normalize to get probabilities
    probabilities = exp_z / np.sum(exp_z, axis=1, keepdims=True)

    return probabilities.squeeze() # Remove extra dimension if input was 1D

# --- Example Usage ---
# Works for small numbers
logits = np.array([2.0, 1.0, 0.1])
print(f"Softmax on small logits:\n{softmax(logits)}\n")

# Works for large numbers without overflow
logits_large = np.array([1000, 1010, 990])
print(f"Softmax on large logits:\n{softmax(logits_large)}")


In [ ]:
import numpy as np

logits = np.array([2.0, 1.0, 0.1])
print(logits.shape)

if logits.ndim == 1:
    logits = logits.reshape(1, -1) # (2D with 1 row, -1 for infer cols)

print(logits.shape)
# axis=1 -> across columns, per row. keepdims -> keeps shape (1,1) for broadcasting
max_logits = np.max(logits, axis=1, keepdims=True)
print(max_logits.shape)
print(max_logits.size)

In [ ]:
z = np.array([[2.0, 1.0, 0.1]])  # shape (1, 3)
print(np.max(z, axis=1).squeeze())         # max of the row

In [ ]:
# axis=1: across columns, i.e. per row (max in each row). axis=0: down rows, per column.
z = np.array([[2.0, 1.0, 0.1],
              [5.0, 3.0, 4.0]])  # shape (2, 3)
print(np.max(z, axis=1).squeeze())         # max of each row

In [ ]:
x = np.array([2.0, 1.0, 0.1])
print(x.shape)

softmax = np.exp(x) / np.exp(x).sum(axis=0, keepdims=True)
print(softmax)

In [ ]:
import torch

x = torch.tensor([2.0, 1.0, 0.1])  # shape (3,)

# softmax across classes; only valid axis for a 1D tensor is dim=0
softmax = torch.exp(x) / torch.exp(x).sum(dim=0, keepdim=True)
print(softmax)

In [ ]:
import torch

x = torch.tensor([[2.0, 1.0, 0.1],
                 [1.0, 3.0, 0.2]])  # shape (2, 3)

# softmax per row (per sample, across its classes) -> dim=1
softmax = torch.exp(x) / torch.exp(x).sum(dim=1, keepdim=True)
print(softmax)

## Linear Regression

#### 0. Core idea

Fit y = w*x + b, a straight line (or hyperplane in higher dimensions), by minimizing MSE (the loss functions notebook covers MSE's formula and gradient in detail).

Toy setup, 1 feature, distance in km predicting delivery time in minutes:
```
x: [1, 2, 3, 4]
y: [15, 25, 35, 40]
```


#### 1. Closed-form solution (least squares)

For 1 feature: slope = sum((x_i-xbar)(y_i-ybar)) / sum((x_i-xbar)^2), intercept = ybar - slope*xbar. This is the exact minimizer of MSE, no iteration needed, a direct formula (the multivariate version is the normal equation, w = (X^T X)^-1 X^T y).

Worked:
```
xbar = (1+2+3+4)/4 = 2.5
ybar = (15+25+35+40)/4 = 28.75

deviations x: [-1.5, -0.5, 0.5, 1.5]
deviations y: [-13.75, -3.75, 6.25, 11.25]

products:     [20.625, 1.875, 3.125, 16.875]  sum = 42.5
sq dev x:     [2.25, 0.25, 0.25, 2.25]         sum = 5.0

slope = 42.5 / 5.0 = 8.5
intercept = 28.75 - 8.5*2.5 = 7.5
```
Fitted line: y = 8.5x + 7.5. Check against actual data: x=1 predicts 16.0 (actual 15), x=4 predicts 41.5 (actual 40), close but not exact, a straight line cannot pass through all 4 points, it finds the best compromise under MSE.


In [ ]:
import numpy as np

x = np.array([1, 2, 3, 4])
y = np.array([15, 25, 35, 40])

xbar, ybar = x.mean(), y.mean()
slope = np.sum((x - xbar) * (y - ybar)) / np.sum((x - xbar) ** 2)
intercept = ybar - slope * xbar

print(f"slope={slope}, intercept={intercept}")
print("predictions:", slope * x + intercept)

# normal equation, generalizes to any number of features
X_design = np.column_stack([np.ones(len(x)), x])  # add intercept column
w = np.linalg.inv(X_design.T @ X_design) @ X_design.T @ y
print("\nnormal equation [intercept, slope]:", w)

#### 2. Gradient descent alternative

The normal equation requires inverting X^T X, an O(p^3) operation in the number of features p, expensive or unstable when p is large or features are highly correlated (the matrix becomes near-singular). Gradient descent avoids the inversion, uses MSE's gradient directly (dL/dw = -2*error*x per point, same gradient shown in the loss functions notebook), iterating: w = w - lr * gradient, same update loop as every other model in this series.

Both converge to the identical answer for plain linear regression (MSE is convex, one global minimum, no local minima to get stuck in), gradient descent is just the more scalable route when the closed form gets expensive.

#### 3. Assumptions

- Linearity: the true relationship between features and target is actually linear (or close enough). Violated relationships (a curve, a threshold effect) get systematically mismodeled, not just noisily mismodeled.
- Independence of errors: residuals should not be correlated with each other, violated in time series data where today's error predicts tomorrow's.
- Homoscedasticity: residual variance should be constant across the range of predictions, not fanning out or shrinking, covered with a worked example in the eval-metrics notebook's residual analysis section.
- No severe multicollinearity: features should not be near-duplicates of each other, otherwise the normal equation becomes numerically unstable and coefficients become uninterpretable (they can swing wildly between correlated features without changing the fit).


In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(x.reshape(-1, 1), y)
print("sklearn slope:", model.coef_, "intercept:", model.intercept_)

# gradient descent from scratch, should converge to the same slope/intercept
w, b = 0.0, 0.0
lr = 0.01
for step in range(2000):
    pred = w * x + b
    error = y - pred
    grad_w = -2 * np.mean(error * x)
    grad_b = -2 * np.mean(error)
    w -= lr * grad_w
    b -= lr * grad_b

print("gradient descent result: slope={:.3f}, intercept={:.3f}".format(w, b))

## SVM

#### 0. Core idea

Find the decision boundary that maximizes the margin, the distance to the nearest point of each class, not just any boundary that separates the classes. Loss function is hinge loss, covered in the loss functions notebook, with the same margin=1 mechanic.

Toy setup, 2D, linearly separable:
```
class +1: (2,2), (3,3), (3,1)
class -1: (0,0), (1,0), (0,1)
```


#### 1. Margin and support vectors

Decision boundary: w.x + b = 0. Margin width = 2/||w||, so maximizing margin means minimizing ||w|| (with the constraint that every point is correctly classified with at least margin 1, the same margin=1 boundary from hinge loss).

Support vectors are the points that end up exactly on the margin boundary, the closest points from each class. Every other point could move around (as long as it stays on the correct side, outside the margin) without changing the solution at all, only the support vectors determine w and b.

This is the key practical property: SVM's decision boundary depends on a small subset of the training data, not all of it. Contrast with logistic regression, which uses every point's gradient contribution to shape the boundary, even points far from it.

#### 2. Hard margin vs soft margin

Hard margin: zero tolerance, every point must be outside the margin and correctly classified. Fails if data is not perfectly linearly separable, or is sensitive to a single noisy point close to the boundary.

Soft margin: allow some points inside the margin or misclassified, penalized by hinge loss (from the loss functions notebook), controlled by parameter C. Large C: heavily penalize margin violations, closer to hard margin, tighter fit, more prone to overfitting on noisy points. Small C: tolerate more violations, wider margin, more regularized.


In [ ]:
import numpy as np
from sklearn.svm import SVC

X = np.array([[2, 2], [3, 3], [3, 1], [0, 0], [1, 0], [0, 1]])
y = [1, 1, 1, -1, -1, -1]

svm = SVC(kernel="linear", C=1000)  # large C, close to hard margin
svm.fit(X, y)

print("support vectors:", svm.support_vectors_)
print("number of support vectors:", len(svm.support_vectors_), "of", len(X), "total points")
print("w:", svm.coef_, "b:", svm.intercept_)
print("margin width (2/||w||):", 2 / np.linalg.norm(svm.coef_))

#### 3. Kernel trick

Not every dataset is linearly separable in its original feature space. The kernel trick maps data into a higher-dimensional space where it becomes separable, without ever explicitly computing the mapping, only the dot products between mapped points (which is all SVM's math actually needs).

Worked example, XOR pattern, not linearly separable in 2D:
```
(0,0) -> class -1
(1,1) -> class -1
(0,1) -> class +1
(1,0) -> class +1
```
No straight line in 2D separates these, the two classes sit on opposite diagonals. Add one engineered feature, z = x1 + x2 - 2*x1*x2 (this is literally the XOR function computed in real-valued arithmetic):
```
(0,0): z = 0+0-2*0*0 = 0
(1,1): z = 1+1-2*1*1 = 0
(0,1): z = 0+1-2*0*1 = 1
(1,0): z = 1+0-2*1*0 = 1
```
Now z=0 exactly matches class -1 and z=1 exactly matches class +1, perfectly separable with the single threshold z=0.5, in a space this specific feature engineering created. A polynomial or RBF kernel does not hand-craft one feature like this, it implicitly searches a much larger (for RBF, infinite-dimensional) space of such transformations, which is why it can solve XOR-like problems no linear boundary in the original space can.


In [ ]:
X_xor = np.array([[0, 0], [1, 1], [0, 1], [1, 0]])
y_xor = [-1, -1, 1, 1]

# linear SVM fails on raw XOR features
linear_svm = SVC(kernel="linear")
linear_svm.fit(X_xor, y_xor)
print("linear kernel accuracy on XOR:", linear_svm.score(X_xor, y_xor))

# RBF kernel handles it via the implicit higher-dimensional mapping
rbf_svm = SVC(kernel="rbf")
rbf_svm.fit(X_xor, y_xor)
print("RBF kernel accuracy on XOR:", rbf_svm.score(X_xor, y_xor))

# manual feature engineering also fixes it, confirming the mechanism
z = X_xor[:, 0] + X_xor[:, 1] - 2 * X_xor[:, 0] * X_xor[:, 1]
print("engineered feature z:", z, "| separable with threshold z=0.5")

#### 4. Feature scaling, assumptions, and kernel-choice tradeoffs

Feature scaling is required, same reason as KNN below: SVM's margin (2/||w||) and the RBF kernel's distance computation are both scale-sensitive, an unscaled large-range feature (e.g. amount_lost in the thousands) would dominate the margin/kernel calculation purely from magnitude, not genuine importance. Always standardize features before fitting an SVM, unlike tree-based models which never need this.

Assumptions: the margin-maximization objective implicitly assumes the classes are separable with a reasonably clean boundary (hard or soft) in SOME feature space, if the classes are genuinely, irreducibly overlapping (not just noisy but truly overlapping distributions), no kernel choice fixes that, more C or a fancier kernel just overfits to noise trying to force a separation that is not really there.

Kernel choice tradeoff:
```
linear:  fastest, most interpretable (w directly weights each feature), only
         works if genuinely linearly separable, like the fraud project's own
         result, logreg (linear) beat every tree/kernel model on sparse TF-IDF
poly:    can capture specific-degree feature interactions explicitly (degree=2
         captures pairwise products), fast for low degree, blows up in
         dimensionality and overfitting risk for high degree
rbf:     most flexible, implicitly infinite-dimensional (the XOR example
         above), but the least interpretable, and gamma (kernel bandwidth)
         needs careful tuning, too high overfits (boundary hugs individual
         points), too low underfits (boundary too smooth/simple)
```
Rule of thumb: start linear (cheap, interpretable, and a strong baseline exactly as often as it was in this series' own tree-vs-linear comparisons), only reach for RBF once linear demonstrably underperforms and you have enough data to tune gamma/C without just overfitting to noise.

## KNN

#### 0. Core idea

No training phase in the usual sense, KNN just stores the training data. At prediction time, find the k closest stored points to the new point, vote (classification) or average (regression) among them. A lazy learner, all the work happens at inference, not training.

Toy setup, 2D, mentions_IRS and urgency_language, same style as the tree notebooks:
```
doc1: (1, 1), gov_impersonation
doc2: (1, 0), gov_impersonation
doc3: (0, 1), romance_scam
doc4: (0, 0), romance_scam
doc5: (0.6, 0.7), unlabeled, predict this one
```


#### 1. Distance metrics

Euclidean distance, most common: d = sqrt(sum((x_i - y_i)^2))

Worked distances from doc5 (0.6, 0.7) to every other point:
```
to doc1 (1,1): sqrt(0.4^2 + 0.3^2) = sqrt(0.16+0.09) = sqrt(0.25) = 0.500
to doc2 (1,0): sqrt(0.4^2 + 0.7^2) = sqrt(0.16+0.49) = sqrt(0.65) = 0.806
to doc3 (0,1): sqrt(0.6^2 + 0.3^2) = sqrt(0.36+0.09) = sqrt(0.45) = 0.671
to doc4 (0,0): sqrt(0.6^2 + 0.7^2) = sqrt(0.36+0.49) = sqrt(0.85) = 0.922
```
Sorted nearest to farthest: doc1 (gov, 0.500), doc3 (romance, 0.671), doc2 (gov, 0.806), doc4 (romance, 0.922).

Manhattan distance, sum of absolute differences instead of squared: d = sum(|x_i - y_i|). Less sensitive to a single large difference in one dimension than Euclidean (no squaring), sometimes preferred in high dimensions or with grid-like data.


#### 2. Voting and choosing k

k=1: only the nearest point matters. Nearest to doc5 is doc1 (gov_impersonation, distance 0.500) -> predict gov_impersonation.

k=3: the 3 nearest are doc1 (gov), doc3 (romance), doc2 (gov) -> 2 votes gov, 1 vote romance -> predict gov_impersonation (majority holds here, but notice it was closer than it looked, doc3 the second-closest point was actually romance).

k too small (like k=1): high variance, a single noisy or mislabeled neighbor flips the prediction entirely, decision boundary is jagged and overfit to individual points.
k too large: high bias, starts averaging in points that are not really representative of the local neighborhood, decision boundary gets smoother but less accurate, at the extreme (k = all points) it just predicts the majority class everywhere, ignoring position entirely.

#### 3. Feature scaling matters here, unlike trees

Distance calculations are dominated by whichever feature has the largest raw scale. If amount_lost (values in the thousands) were combined with urgency_language (0 or 1) without scaling, amount_lost alone would decide every distance, urgency_language's contribution would be numerically drowned out.

Trees (Random Forest, XGBoost, and friends) split one feature at a time by threshold, so scale never matters there, this is a KNN-specific (and SVM/logreg-specific, anything distance or gradient based) requirement.

#### 4. Curse of dimensionality

As dimensions grow, all points start looking roughly equidistant from each other, the ratio between the nearest and farthest neighbor's distance approaches 1. "Nearest neighbor" stops being a meaningfully informative concept in very high-dimensional sparse spaces (like raw TF-IDF with thousands of columns), which is part of why KNN is rarely the first choice for high-dimensional text classification, unlike logreg or tree ensembles.


In [ ]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

X = np.array([[1, 1], [1, 0], [0, 1], [0, 0]])
y = ["gov", "gov", "romance", "romance"]
new_point = [[0.6, 0.7]]

distances = np.linalg.norm(X - new_point, axis=1)
for label, dist in sorted(zip(y, distances), key=lambda t: t[1]):
    print(f"{label}: distance={dist:.3f}")

for k in [1, 3]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X, y)
    print(f"\nk={k} prediction:", knn.predict(new_point))

## Naive Bayes

#### 0. Core idea

Bayes theorem: P(class|words) proportional to P(class) * P(words|class). The "naive" part: assume every word is conditionally independent given the class, so P(words|class) = product of P(each word|class) individually. Almost never literally true (word order and co-occurrence obviously matter), but the approximation works well in practice for text classification, and makes the math trivially cheap, just counting.

Toy training set, 2 classes, 2 tiny documents each:
```
gov_impersonation: "IRS money arrest", "IRS pay now"
romance_scam: "love money wire", "heart money please"
```
Word counts per class: gov = {IRS:2, money:1, arrest:1, pay:1, now:1}, total 6 words.
romance = {love:1, money:2, wire:1, heart:1, please:1}, total 6 words.
Vocabulary (union of all unique words): IRS, money, arrest, pay, now, love, wire, heart, please, size 9.
Priors: P(gov) = P(romance) = 0.5 (2 docs each out of 4 total).


#### 1. Classifying a new document, and the zero-probability problem

New document: "IRS money". Compute P(gov|doc) and P(romance|doc), unnormalized (proportional only, skip the shared denominator).

Without smoothing, P(word|class) = count(word,class) / total_words_class:
```
P(IRS|gov) = 2/6 = 0.333,  P(money|gov) = 1/6 = 0.167
score(gov) = P(gov) * P(IRS|gov) * P(money|gov) = 0.5 * 0.333 * 0.167 = 0.0278

P(IRS|romance) = 0/6 = 0  <- "IRS" never appeared in any romance_scam doc
score(romance) = 0.5 * 0 * P(money|romance) = 0
```
romance's score is exactly zero, not because romance is truly impossible, but because "IRS" simply never appeared in the tiny romance training set. One unseen word wipes out the entire product, that is the zero-probability problem, and it gets worse the more words a real document has (more chances to hit an unseen one).


#### 2. Laplace smoothing: the fix

Add 1 to every count, and V (vocabulary size) to every denominator: P(word|class) = (count(word,class)+1) / (total_words_class + V).

```
V = 9

P(IRS|gov)     = (2+1)/(6+9) = 3/15   = 0.200
P(money|gov)   = (1+1)/(6+9) = 2/15   = 0.133
score(gov)     = 0.5 * 0.200 * 0.133 = 0.0133

P(IRS|romance)   = (0+1)/(6+9) = 1/15 = 0.0667
P(money|romance) = (2+1)/(6+9) = 3/15 = 0.200
score(romance)   = 0.5 * 0.0667 * 0.200 = 0.00667
```
Now both scores are nonzero, and comparable: score(gov)=0.0133 beats score(romance)=0.00667, predict gov_impersonation, the same conclusion, but arrived at honestly instead of by one word forcing a hard zero.


In [ ]:
from collections import Counter

gov_docs = ["IRS money arrest", "IRS pay now"]
romance_docs = ["love money wire", "heart money please"]

gov_counts = Counter(" ".join(gov_docs).split())
romance_counts = Counter(" ".join(romance_docs).split())
vocab = set(gov_counts) | set(romance_counts)
V = len(vocab)
total_gov = sum(gov_counts.values())
total_romance = sum(romance_counts.values())

def smoothed_prob(word, counts, total, V):
    return (counts.get(word, 0) + 1) / (total + V)

def score(doc_words, counts, total, prior, V):
    p = prior
    for w in doc_words:
        p *= smoothed_prob(w, counts, total, V)
    return p

new_doc = ["IRS", "money"]
score_gov = score(new_doc, gov_counts, total_gov, 0.5, V)
score_romance = score(new_doc, romance_counts, total_romance, 0.5, V)

print("score(gov):", score_gov)
print("score(romance):", score_romance)
print("prediction:", "gov_impersonation" if score_gov > score_romance else "romance_scam")

#### 3. Compare against sklearn's MultinomialNB, and when to use it

MultinomialNB uses the same Laplace-smoothed counting, applied via TF-IDF or raw counts rather than a hand-built dict.

When it is a good choice: text classification with limited data, since it needs no gradient descent or iterative tuning, just counting, so it trains near-instantly and works reasonably even with few examples per class. Where it tends to lose: whenever features are correlated (the independence assumption is badly wrong), logreg or a tree model usually wins once you have enough data for them to actually learn those correlations.


In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer

docs = gov_docs + romance_docs
labels = ["gov", "gov", "romance", "romance"]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(docs)

nb = MultinomialNB(alpha=1.0)  # alpha=1.0 is Laplace smoothing
nb.fit(X, labels)

new_doc_vec = vectorizer.transform(["IRS money"])
print("prediction:", nb.predict(new_doc_vec))
print("class probabilities:", nb.predict_proba(new_doc_vec))